In [2]:
import pyspark
pyspark .__file__

'/home/dhirajtripathi93/data-engineering/spark/.venv/lib/python3.10/site-packages/pyspark/__init__.py'

In [6]:
!uv pip install pyspark==3.3.2

Using Python 3.10.12 environment at: /home/dhirajtripathi93/data-engineering/spark/.venv
Resolved 2 packages in 5.26s                                         
Prepared 2 packages in 16.53s                                            
Uninstalled 2 packages in 33ms
Installed 2 packages in 19ms                                
 - py4j==0.10.9.9
 + py4j==0.10.9.5
 - pyspark==4.2.1
 + pyspark==3.3.2


In [3]:
!uv pip show pyspark

Using Python 3.10.12 environment at: /home/dhirajtripathi93/data-engineering/spark/.venv
Name: pyspark
Version: 3.3.2
Location: /home/dhirajtripathi93/data-engineering/spark/.venv/lib/python3.10/site-packages
Requires: py4j
Required-by: spark


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/07 12:09:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-11-06 14:35:06--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.26.156.215
Connecting to github.com (github.com)|20.26.156.215|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-06T15%3A09%3A03Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-11-06T14%3A08%3A34Z&ske=2026-11-06T15%3A09%3A03Z&sks=b&skv=2018-11-09&sig=LThQkb1xQYwI58RzBL6T05Kt6mOa7vgR10zZsbHPsF8%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5Mzk3OTE2NCwibmJmIjoxNzkzOTc1NTY0LCJwYXRoIj

In [12]:
!gunzip fhvhv_tripdata_2021-01.csv.gz

In [13]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [17]:
!head -n 101 fhvhv_tripdata_2021-01.csv > head.csv
!wc -l head.csv


101 head.csv


In [8]:
!uv pip install pandas

Using Python 3.10.12 environment at: /home/dhirajtripathi93/data-engineering/spark/.venv
Resolved 6 packages in 205ms                                         
Prepared 3 packages in 382ms                                             
Installed 3 packages in 43ms                                
 + numpy==2.2.6
 + pandas==2.3.3
 + pytz==2026.6


In [9]:
import pandas as pd
df_pandas = pd.read_csv('head.csv')
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [14]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [5]:
from pyspark.sql import types


In [6]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [7]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [8]:
df = df.repartition(24)


In [9]:
df.write.parquet('fhvhv/2021/01/',mode='overwrite')


In [10]:
df = spark.read.parquet('fhvhv/2021/01')

In [11]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [10]:
df.select('pickup_datetime', 'dropoff_datetime',
          'PULocationID', 'DOLocationID')\
.filter(df.hvfhs_license_num=='HV0003')\
.show()

[Stage 1:>                                                          (0 + 1) / 1]

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-05 22:14:07|2021-01-05 22:32:28|         189|         107|
|2021-01-02 17:59:55|2021-01-02 18:10:39|          88|         137|
|2021-01-02 23:57:54|2021-01-03 00:15:48|         238|         224|
|2021-01-06 15:53:13|2021-01-06 16:07:07|         169|         208|
|2021-01-07 07:35:24|2021-01-07 07:55:49|          75|          88|
|2021-01-07 08:45:12|2021-01-07 08:51:17|         210|         210|
|2021-01-02 15:44:26|2021-01-02 16:10:50|         243|          69|
|2021-01-04 16:50:28|2021-01-04 16:57:43|         250|         213|
|2021-01-03 10:30:34|2021-01-03 10:44:53|          87|          79|
|2021-01-03 22:05:20|2021-01-03 22:27:55|          68|         181|
|2021-01-04 08:01:02|2021-01-04 08:33:27|          95|         236|
|2021-01-02 13:01:10|2021-01-02 13:08:11|       

In [12]:
from pyspark.sql import functions as F

In [13]:
def crazy_stuff(base_num):
    if not base_num or len(base_num)<=1:
        return None

    try:
        num = int(base_num[1:])
        if num % 7 == 0:
            return f's/{num:03x}'
        elif num % 3 == 0:
            return f'a/{num:03x}'
        else:
            return f'e/{num:03x}'

    except ValueError:
        return None
    
    

In [14]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType = types.StringType())

In [22]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime))\
.withColumn('dropoff_date', F.to_date(df.dropoff_datetime))

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string, pickup_date: date, dropoff_date: date]

In [16]:
df2 = df\
.withColumn('hexabase_id',crazy_stuff_udf(df.dispatching_base_num))\
.withColumn('pickup_date', F.to_date(df.pickup_datetime))\
.withColumn('dropoff_date', F.to_date(df.dropoff_datetime))\
.select('hexabase_id', 'pickup_date', 'dropoff_date','PULocationID', 'DOLocationID') \
.show()

[Stage 4:>                                                          (0 + 1) / 1]

+-----------+-----------+------------+------------+------------+
|hexabase_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+-----------+------------+------------+------------+
|      e/9ce| 2021-01-03|  2021-01-03|         255|          34|
|      e/b42| 2021-01-05|  2021-01-05|         189|         107|
|      e/b33| 2021-01-02|  2021-01-02|          88|         137|
|      e/b38| 2021-01-02|  2021-01-03|         238|         224|
|      e/b3b| 2021-01-06|  2021-01-06|         169|         208|
|      e/b33| 2021-01-07|  2021-01-07|          75|          88|
|      e/acc| 2021-01-07|  2021-01-07|         210|         210|
|      e/acc| 2021-01-02|  2021-01-02|         243|          69|
|      e/b35| 2021-01-04|  2021-01-04|         250|         213|
|      s/b3d| 2021-01-03|  2021-01-03|          87|          79|
|      e/a39| 2021-01-03|  2021-01-03|          68|         181|
|      s/acd| 2021-01-04|  2021-01-04|          95|         236|
|      s/b13| 2021-01-02|